# Linking People Across Datasets

A simple record linkage example using the [Splink](https://moj-analytical-services.github.io/splink/) linkage framework, looking at various stages of the data linkage pipeline, identifying individuals across two datasets using personal identifiers and a probabilistic model.


## Imports
Here we load in the external functionality used in this notebook.

It's mostly Splink functionality, plus the IPython IFrame class to allow us to render HTML dashboards within the notebook.

In [1]:
from splink import splink_datasets, Linker, SettingsCreator
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
)
from splink.blocking_rule_library import block_on, CustomRule
from splink import comparison_library as cl
from splink.exploratory import completeness_chart, profile_columns


from IPython.display import IFrame  # To render html inside the notebook

In [2]:
from sqlalchemy import create_engine
from splink import TrinoAPI
import pandas as pd

engine = create_engine(
    "trino://splink@localhost:8080/iceberg/splink_test"
)

my_db_api = TrinoAPI(
    engine,
    catalog="iceberg",
    schema="splink_test",
    schema_location="s3a://splink/"
)

df = pd.DataFrame({
    "id": [1,2,3],
    "name": ["a","b","c"]
})

my_db_api._table_registration(df, "test_table")

print(my_db_api.table_exists_in_database("test_table"))

True


# BACKEND
Alex, I presume here we will import the Trino backend

In [3]:
# from splink import TrinoAPI

# my_db_api = TrinoAPI()

## Load in the toy dataset and split it into two datasets
Splink contains a few example datasets, in this example we'll use the `fake_1000` dataset, containing `first_name`, `surname`, `dob`, `city`, and `email` information.
There is also a `cluster` column, which is a ground-truth label as to the entity (cluster/person) the record belongs to, however we won't use this in our model.

We first load in the dataset and then divide it into separate dataframes to simulate having two source datasets. We achieve this with a simple 50% random sampling.

In [4]:
# Load in the fake_1000 dataset
df = splink_datasets.fake_1000

# Randomly sample 50% of the records for the second dataset
df_l = df.sample(frac=0.5, random_state=42)
df_r = df.drop(df_l.index)

df_l.head(5)

,unique_id,first_name,surname,dob,city,email,cluster
521,521,George,Owen,1980-06-29,London,go94@bradshaw-sanders.com,129
737,737,NaN,Ahmed,2005-09-16,London,oa14@lee.com,187
740,740,Ahmed,Oscar,2005-09-18,NaN,oa14@lee.coom,187
660,660,Charlie,Hall,2018-06-26,Biriminghm,NaN,168
411,411,Emily,Atkinson,2008-05-05,NaN,emilya@roberts.com,106


## Profiling the data
You should understand the data you're working with, so lets take a look at the datasets we have.

We'll look at the amount of **Missingness** (NULL values) per column, per dataset:
* High missingness might mean that a certain piece of information is less useful in identifying matches as we often treat NULL values as neutral (neither negative nor positive) in determining the overall score when comparing records.

We'll also look at distributions of values within the columns, specifically looking at **Cardinality** and **Skew**:

* High cardinality (e.g. DOB) is beneficial, as it is less likely that non-matches randomly share a value. Low cardinality (e.g. Sex) is less useful.
* Skew of the values within a field can also affect the information gained. For example `city` has a high skew towards the value "London", and so observing this value has less informative power than other cities.


In [5]:
completeness_chart(
    [df_l, df_r],
    cols=["first_name", "surname", "dob", "city", "email"],
    db_api=my_db_api,
    table_names_for_chart=["df_left", "df_right"],
)

alt.LayerChart(...)

In [6]:
profile_columns([df_l, df_r], db_api=my_db_api)

alt.VConcatChart(...)

## Define the blocking rules to generate potential edges
In order to match records we need to generate record pairs to compare, however doing this naively is incredibly inefficient.

If we just generate every possible pair we end up generating $\frac{n\left(n-1\right)}2$ possible comparisons. For this dataset we would have ~500,000 comparisons to evaluate which isn't terrible, but for a modest dataset of 1 million records we would have ~500 billion pairs to compare...

To get around this naivety, we define some **Blocking rules** to constrain the possible pairs we wish to even consider. These blocking rules are a list of rules with which to filter the links we're going to look at. For an edge to be considered for scoring it needs to match to at least one of these rules, e.g. If we have a rule that our left and right records must match on first name (`l.first_name = r.first_name`) then "Jane Doe" and "Jane Smith" will be considered for scoring, but "Jane Doe" and "John Doe" will not.

In [7]:
blocking_rules_to_generate_predictions = [
    block_on("first_name"),
    block_on("surname"),
    block_on("dob"),
    # HAD TO CHANGE THIS TO TRINO SYNTAX
    CustomRule("substr(l.first_name, 1, 1) = substr(r.first_name, 1, 1) AND l.city = r.city"),
]

cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=[df_l, df_r],
    blocking_rules=blocking_rules_to_generate_predictions,
    link_type="link_and_dedupe",
    db_api=my_db_api,
)

alt.Chart(...)

## Define the comparison methods to score record pairs
Once we've defined which record pairs we want to consider for scoring, we need to define **how** we will score them.

To do this we need to define methods of information scoring. For strings this may be similarity scores or edit distance functions (e.g. one character change, Jaro-Winkler similarity > 0.9 etc.), for dates it might be distance based (e.g. within 1 month, within 1 year etc.).

In the Splink framework, we can use pre-defined comparison methods for a variety of different information types, including for names, dates of birth, emails etc. We can also define our own comparison methods, but this is more advanced.

More information on the available comparisons within Splink can be found in [the documentation](https://moj-analytical-services.github.io/splink/api_docs/comparison_library.html).

In [8]:
comparisons = [
    cl.NameComparison(
        "first_name",
    ),
    cl.NameComparison("surname"),
    cl.DateOfBirthComparison(
        "dob",
        input_is_string=True,
        invalid_dates_as_null=True,
    ),
    cl.ExactMatch("city").configure(term_frequency_adjustments=True),
    cl.EmailComparison("email"),
]

## Instantiate a Linker object
Now we bring the blocking rules and comparison methods together into a `Linker` object. This lets us call methods for training the model, predicting edge scores, and clustering the results to obtain a joining key.

The `Linker` object takes in the datasets, the settings for the linkage methods, and a backend database API (here we use [DuckDB](https://duckdb.org/)).

In [9]:
linker = Linker(
    [df_l, df_r],
    settings=SettingsCreator(
        link_type="link_and_dedupe",
        blocking_rules_to_generate_predictions=blocking_rules_to_generate_predictions,
        comparisons=comparisons,
        retain_matching_columns=True,
        retain_intermediate_calculation_columns=True,
    ),
    db_api=my_db_api,
    input_table_aliases=["df_left", "df_right"],
)

## Training the probabilistic linkage model
Training a Felegi-Sunter model requires 3 parameters to score a given record pair:
1. $\lambda$ - the prior probability that two random records are a match
2. $u$ values - the probability that given an observed comparison scenario a record pair **is not a match**
3. $m$ values - the probability that given an observed comparison scenario a record pair **is a match**

## Estimating $\lambda$
To estimate the prior probability, $\text{Pr}(\text{match})$, we provide a set of rules which we believe will contain true matches, and then see how likely we are to observe these rules.


In [10]:
# had to add _distance to lev
deterministic_rules = [
    "l.first_name = r.first_name and levenshtein_distance(r.dob, l.dob) <= 1",
    "l.surname = r.surname and levenshtein_distance(r.dob, l.dob) <= 1",
    "l.first_name = r.first_name and levenshtein_distance(r.surname, l.surname) <= 2",
    "l.email = r.email",
]


linker.training.estimate_probability_two_random_records_match(
    deterministic_rules, recall=0.7
)

Probability two random records match is estimated to be  0.00333.
This means that amongst all possible pairwise record comparisons, one in 300.13 are expected to match.  With 499,500 total possible comparisons, we expect a total of around 1,664.29 matching pairs


## Estimating $u$ value
To estimate the $u$ values, we can randomly sample record pairs and compute $\text{Pr}(\text{comparison}|\text{non-match})$. The more samples we make, the more accurate our estimate of $u$ values, but at the cost of it taking longer to run.

In [11]:
linker.training.estimate_u_using_random_sampling(max_pairs=1e6, seed=1)

You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - first_name (no m values are trained).
    - surname (no m values are trained).
    - dob (no m values are trained).
    - city (no m values are trained).
    - email (no m values are trained).


## Estimating $m$ values

To estimate the $m$ values, we need to compute $\text{Pr}(\text{comparison}|\text{match})$, however we don't actually **know** the set of matches (because if we did have these labels, we wouldn't need to perform linkage).

Instead we can use the Expectation Maximisation algorithm to estimate the $m$ values by fixing certain assumptions and then observing the behaviour of the other comparison scenarios. i.e. if we look at the comparisons where `dob` is the same, how do the comparisons on `first_name`, `surname`, `city` and `email` vary? What about if we look at comparisons where `email` is the same?

In Splink this is achieved by calling `estimate_parameters_using_expectation_maximisation()` on a rule that produces a sample of matches without being too restrictive. The EM algorithm iteratively updates model parameters by first predicting which record pairs are matches (Expectation), and then using this predictions to re-estimate the $m$ values to improve the next round of predictions (Maximisation).

In this example we have three training sessions to estimate $m$ values, and then our final model is the average of these estimates.

In [12]:
session_dob = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("dob")
)
session_email = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("email")
)
session_first_name = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("first_name")
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."dob" = r."dob"

Parameter estimates will be made for the following comparison(s):
    - first_name
    - surname
    - city
    - email

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - dob

Level Jaro-Winkler >0.88 on username on comparison email not observed in dataset, unable to train m value

Iteration 1: Largest change in params was -0.405 in the m_probability of surname, level `Exact match on surname`
Iteration 2: Largest change in params was 0.111 in probability_two_random_records_match
Iteration 3: Largest change in params was 0.0486 in the m_probability of first_name, level `All other comparisons`
Iteration 4: Largest change in params was 0.0177 in probability_two_random_records_match
Iteration 5: Largest change in params was 0.00741 in probability_two_random_records_match
Iteration 6: Largest change in pa

## Visualise model training sessions
This visualisation will show us how our different training schemes estimate the $m$ values. Notice how there are some differences in the estimated values, why might that be the case based on the training scheme used?

In [13]:
linker.visualisations.parameter_estimate_comparisons_chart(include_u=True)

alt.Chart(...)

## Visualise final model parameters
The first visualisation shows how the $m$ and $u$ values behave for each comparison method we have defined in our model.

In [14]:
linker.visualisations.m_u_parameters_chart()

alt.HConcatChart(...)

The following visualisation shows how the model will weight observations based on which comparison level they hit.
This is how we will obtain a final score for a given record pair, e.g. a comparison of two `first_names` "john" vs "john" will hit the "Exact match" level on `first_name` and be assigned the match weight ($\log2(m/u)$) of that level.

In [15]:
linker.visualisations.match_weights_chart()  # This might throw a "UserWarning" for Altair that can be ignored.

/Users/alex/Devel/splink/.venv/lib/python3.14/site-packages/altair/vegalite/v6/api.py:4124: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

## Visualise how term-frequency adjustments impact various columns in our model
When we looked at Skew earlier, we can see that `first_name`, `surname`, and `city` are all highly skewed distributions. To account for this in our model we can adjust the match weight based on the frequency at which we observe a given value in our dataset. For example "London" is a commonly observed value for `city`, and so we can slightly reduce the score for a match on that value compared to say "Swansea" or "Bristol".

The following plot shows us the top and bottom frequency values observed for the `city` column, and their associated adjustments in the model.

In [16]:
linker.visualisations.tf_adjustment_chart("city")

alt.HConcatChart(...)

## Predict and score the edges
Now that we have a trained model, we can go through the edges our blocking rules identify and score them using the final model parameters using the `linker.inference.predict()` method.

We can also disregard all edges that score below a certain threshold with the `threshold_match_probability` value, here removing edges scored below 50% match probability.

The `predict` method returns a table where each row describes a scored edge pair, including the match weight, match probability, and left and right record descriptors (dataset name, unique id etc.).

In [17]:
df_predict = linker.inference.predict(threshold_match_probability=0.5)

df_predict.as_pandas_dataframe(limit=5)

Blocking time: 0.80 seconds
Predict time: 0.96 seconds


,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,first_name_l,first_name_r,gamma_first_name,tf_first_name_l,...,bf_city,bf_tf_adj_city,email_l,email_r,gamma_email,tf_email_l,tf_email_r,bf_email,bf_tf_adj_email,match_key
0,2.677874,0.864849,df_left,df_right,521,524,George,Owen,0,0.01444,...,1.000000,1.000000,go94@bradshaw-sanders.com,geo94@bradshaw-sanders.com,2,0.002535,0.001267,233.300031,1.000000,2
1,13.785028,0.999929,df_left,df_left,521,522,George,NaN,-1,0.01444,...,1.000000,1.000000,go94@bradshaw-sanders.com,go94@bradshaw-sanders.com,4,0.002535,0.002535,255.161745,0.865482,2
2,14.538694,0.999958,df_left,df_left,521,526,George,George,4,0.01444,...,0.458481,1.000000,go94@bradshaw-sanders.com,go94@brndshaw-saaders.com,3,0.002535,0.001267,209.308071,1.000000,0
3,4.768409,0.964607,df_left,df_left,521,525,George,George,4,0.01444,...,1.000000,1.000000,go94@bradshaw-sanders.com,NaN,-1,0.002535,NaN,1.000000,1.000000,0
4,6.181811,0.986412,df_left,df_left,521,523,George,George,4,0.01444,...,10.277934,0.259162,go94@bradshaw-sanders.com,NaN,-1,0.002535,NaN,1.000000,1.000000,0


## Visualise some of the low- and high-scoring edges
Here we can use a waterfall chart to visualise a single predicted link, showing how the two records have been scored.

We'll take a look at some low and high scoring predictions, using DuckDB to select some examples from both low and high match_probability. This task can help identify if we need to increase the threshold in our `predict()` step, or look back at our blocking rules or comaprisons to refine the model and better predict matches.

In [18]:
sample_of_low_scoring_predictions = linker.misc.query_sql(
    f"""
SELECT *
FROM {df_predict.physical_name}
ORDER BY match_probability ASC
LIMIT 10
""",
    output_type="splink_df",
).as_record_dict()

linker.visualisations.waterfall_chart(sample_of_low_scoring_predictions)

alt.LayerChart(...)

In [19]:
sample_of_high_scoring_predictions = linker.misc.query_sql(
    f"""
SELECT *
FROM {df_predict.physical_name}
ORDER BY match_probability DESC
LIMIT 10
""",
    output_type="splink_df",
).as_record_dict()

linker.visualisations.waterfall_chart(sample_of_high_scoring_predictions)

alt.LayerChart(...)

## Comparison viewer
In this visualisation we look at some representative examples of different comparisons across the predicted edges, again this can be useful in tweaking the model, and help to identify what scenarios we might not be happy with within our final set of identified edges.

In [20]:
linker.visualisations.comparison_viewer_dashboard(
    df_predict,
    out_path="./comparison_dashboard.html",
    overwrite=True,
    num_example_rows=5,
)

IFrame(src="./comparison_dashboard.html", width="100%", height=1200)

## Cluster the edges
Once we have our predicted edges, we can cluster connected records together to identify groups of records predicted to belong to the same person. In this example we use the simplistic [Connected Components](https://en.wikipedia.org/wiki/Component_(graph_theory)) approach, however more complex algorithms from graph literature can be used to further refine the identified clusters/entities/people.

e.g. If we have the edges `[[A,B], [B,C], [D,E]]` then we would have two clusters. The first contains the records `[A,B,C]` and the second contains `[D,E]`

To do this in Splink we use `linker.clustering.cluster_pairwise_predictions_at_threshold()` passing in the predictions obtained from the `predict()` method. This returns a table in which each row allocates a cluster ID to each record in the original input table(s), this cluster ID is the grouping key that joins records together as single entities.


In [21]:
df_cluster = linker.clustering.cluster_pairwise_predictions_at_threshold(df_predict)
df_cluster.as_pandas_dataframe(limit=1)

Completed iteration 1, num edges remaining to process: 468
Completed iteration 2, num edges remaining to process: 16
Completed iteration 3, num edges remaining to process: 0


,cluster_id,unique_id,first_name,surname,dob,city,email,cluster,source_dataset
0,df_right-__-339,339,Rose,Wood,2004-12-21,Readng,rosewood@coleman.org,90,df_right


## Visualise a selection of clusters
This visualisation will show some example clusters in a nice dashboard using the `linker.visualisations.cluster_studio_dashboard()` method.

There are a variety of ways to sample clusters to show:
* randomly sample (default) with `sampling_method='random'`
* sample by cluster size (node count) with `sampling_method='by_cluster_size'`
* pass in cluster_ids directly with a list of strings via the `cluster_ids` argument to look at specific clusters

For more detail, see [the dashboard docs](https://moj-analytical-services.github.io/splink/api_docs/visualisations.html#splink.internals.linker_components.visualisations.LinkerVisualisations.cluster_studio_dashboard).

In [22]:
linker.visualisations.cluster_studio_dashboard(
    df_predict=df_predict,
    df_clustered=df_cluster,
    out_path="./cluster_studio.html",
    overwrite=True,
    sampling_method="random",
    sample_size=30,
)

IFrame(src="./cluster_studio.html", width="100%", height=1200)

## Evaluation
Seeing as we have ground truth labels in the `cluster` column of the dataset, let's look at some of the errors in prediction. In real cases we would need to either use a proxy variable (e.g. a very strong individual identifier such as NHS number / Social Security), or we would need to review samples to create a small labelled set to estimate performance.

In [23]:
linker.evaluation.prediction_errors_from_labels_column("cluster").as_pandas_dataframe()

Blocking time: 0.89 seconds
Predict time: 1.44 seconds


,clerical_match_score,found_by_blocking_rules,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,first_name_l,first_name_r,...,email_l,email_r,gamma_email,tf_email_l,tf_email_r,bf_email,bf_tf_adj_email,cluster_l,cluster_r,match_key
0,1.0,False,-3.003240,0.110889,df_left,df_right,737,742,NaN,Ahmed,...,oa14@lee.com,oa14@lee.com,4,0.005070,0.005070,255.161745,0.432741,187,187,4
1,1.0,False,4.569456,0.959585,df_left,df_left,737,741,NaN,Osrac,...,oa14@lee.com,NaN,-1,0.005070,NaN,1.000000,1.000000,187,187,4
2,1.0,False,3.086025,0.894643,df_left,df_left,737,740,NaN,Ahmed,...,oa14@lee.com,oa14@lee.coom,3,0.005070,0.001267,209.308071,1.000000,187,187,4
3,1.0,False,-3.335440,0.090138,df_left,df_right,740,739,Ahmed,Oscar,...,oa14@lee.coom,oa14@lee.com,3,0.001267,0.005070,209.308071,1.000000,187,187,4
4,1.0,False,5.552446,0.979136,df_left,df_right,740,738,Ahmed,NaN,...,oa14@lee.coom,oa14@lee.com,3,0.001267,0.005070,209.308071,1.000000,187,187,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,1.0,False,6.648913,0.990133,df_right,df_right,214,219,Luca,aucaL,...,lucas@harrington.com,lucas@harrington.com,4,0.003802,0.003802,255.161745,0.576988,57,57,4
665,1.0,False,-9.812465,0.001111,df_right,df_right,214,217,Luca,NaN,...,lucas@harrington.com,NaN,-1,0.003802,NaN,1.000000,1.000000,57,57,4
666,1.0,True,-1.466908,0.265654,df_right,df_right,217,219,NaN,aucaL,...,NaN,lucas@harrington.com,-1,NaN,0.003802,1.000000,1.000000,57,57,2
667,1.0,False,2.765801,0.871816,df_right,df_right,230,232,George,Gerge,...,g.davies36@ford-graves.com,NaN,-1,0.003802,NaN,1.000000,1.000000,60,60,4


Here we show a labelling tool that helps visualise how threshold selection affects various performance metrics. You can vary the proposed match probability threshold and see how it affects the number of false negatives (missed links) and false positives (incorrectly identified links) as well as materics such as [Precision and Recall](https://en.wikipedia.org/wiki/Precision_and_recall)

In [24]:
linker.evaluation.accuracy_analysis_from_labels_column("cluster")

Blocking time: 0.73 seconds
Predict time: 0.13 seconds


alt.HConcatChart(...)

## Clerical Review
We can also see if there are records "close" to unmatched ones, and perform some clerical review to improve the model.

Here we get the details of records which did not find a match (clusters which contain only a single record)

In [25]:
linker.misc.query_sql(
    f"SELECT cluster_id FROM {df_cluster.physical_name} GROUP by cluster_id HAVING count(1) > 2"
)

,cluster_id
0,df_left-__-973
1,df_left-__-237
2,df_left-__-810
3,df_left-__-849
4,df_left-__-695
...,...
160,df_left-__-530
161,df_left-__-826
162,df_left-__-274
163,df_left-__-930


#### Use the labelling tool to look at "close" records

We can then use a unique_id and source dataset name to produce a labelling tool showing "close" records that we can label as either a match or not. This can then feedback into the metric calculations.

In [26]:
linker.evaluation.labelling_tool_for_specific_record(
    unique_id="735",
    source_dataset="df_right",
    overwrite=True,
    match_weight_threshold=-10,
)

IFrame(src="./labelling_tool.html", width="100%", height=1200)


The Splink labelling tool is still in development, which means some features may change and there may be bugs.
Your feedback will help us improve it. Go to
github.com/moj-analytical-services/splink/discussions/new?category=general
to give us feedback.


# Tasks and thought exercises

<u>Datasets</u>
* Think about how data quality can be improved? Are there cleaning techniques we could use?
* Advanced: Currently we use the fields as-is, however there are more complex techniques to represent these records, have a read about match keys, Bloom filters, and vector embeddings.

<u>Blocking Rules</u>
* Think about whether the current rules are too loose (introduce too many false positives).
* Try editing the blocking rules to capture more true links. Use `CustomRule()` to define your own rules.
* Can you think about real-life scenarios that may be biased by rules that are too strict?

<u>Comparisons</u>
* Advanced: Think about whether you can define your own comparison methods.
* Can you use the `Comparison Viewer` dashboard to identify comparison levels that contain both ture and false links?

<u>Prediction</u>
* Can you update the `threshold_match_probability` to capture more true positives. Does it introduce more false positives? How would you deal with this?

<u>Clustering</u>
* Can you generate a sample of cluster IDs of interest, and look at them in the `Cluster Studio` dashboard?
* Advanced: How would you propose we improve on the Connected Components approach?

<u>Clerical Review</u>
* All metrics of quality are **estimates**. What ways could we sample our data to perform clerical review? How can these clerical review processes bias ourestimates of quality?
* How can linkage quality affect the use of linked datasets?